1. Dropout 
实现 dropout_layer 函数，该函数以dropout的概率丢弃张量输入x中的元素

In [3]:
import torch
from torch import nn
from d2l import torch as d2l

def dropout_layer(X, dropout): # dropout是丢弃比率
    assert 0<= dropout <= 1    # 断言？反正是出问题了
    if dropout == 1:  # 返回和X shape一样的数值全0的内容
        return torch.zeros_like(X) 
    if dropout == 0:
        return X
    mask = (torch.randn(X.shape) > dropout).float()  # 生成0-1均匀分布的概率 是丢弃的概率结合下一句来看
    return mask * X / (1.0 - dropout)

测试 dropout_layer 函数

In [5]:
X = torch.arange(16, dtype=torch.float32).reshape((2, 8))
print(X)
print(dropout_layer(X, 0.))
print(dropout_layer(X, 0.5))
print(dropout_layer(X, 1.))

tensor([[ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11., 12., 13., 14., 15.]])
tensor([[ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11., 12., 13., 14., 15.]])
tensor([[ 0.,  0.,  0.,  0.,  8.,  0., 12.,  0.],
        [16., 18.,  0.,  0., 24.,  0.,  0.,  0.]])
tensor([[0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.]])


2. 定义具有两个隐藏层的多层感知机，每个隐藏层包含256个单元

In [8]:
num_inputs, num_outputs, num_hiddens1, num_hiddens2 = 784, 10, 256, 256

dropout1, dropout2 = 0.2, 0.5

class Net(nn.Module):
    def __init__(self, num_inputs, num_outputs, num_hiddens1, num_hiddens2,
                is_training=True):
        super(Net, self).__init__()
        self.num_inputs = num_inputs
        self.training = is_training
        self.lin1 = nn.Linear(num_inputs, num_hiddens1)
        self.lin2 = nn.Linear(num_hiddens1, num_hiddens2)
        self.lin3 = nn.Linear(num_hiddens2, num_outputs)
        self.relu = nn.ReLU()
        
    def forward(self, X):
        H1 = self.relu(self.lin1(X.reshape((-1, self.num_inputs)))) # h1是隐藏层1输出
        if self.training == True:
            H1 = dropout_layer(H1, dropout1)
        H2 = self.relu(self.lin2(H1))  # 隐藏层1的输出放到第二个隐藏层中，计算输出H2
        if self.training == True:
            H2 = dropout_layer(H2, dropout2)
        out = self.lin3(H2)
        return out
  
net = Net(num_inputs, num_outputs, num_hiddens1, num_hiddens2)

训练测试同之前

以下是简洁实现

In [13]:
net = nn.Sequential(
    nn.Flatten(), nn.Linear(784, 256), nn.ReLU(),
    nn.Dropout(dropout1), nn.Linear(256, 256), nn.ReLU(),
    nn.Dropout(dropout2), nn.Linear(256, 10))

def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weights, std=0.01)
        
net.apply(init_weights);

AttributeError: 'Linear' object has no attribute 'weights'